In [ ]:
# Ноутбук перенесён в подпапку — восстанавливаем CWD в корень thesis/
import os
from pathlib import Path

_cwd = Path.cwd()
while _cwd.name != "thesis" and _cwd.parent != _cwd:
    _cwd = _cwd.parent
if _cwd.name == "thesis":
    os.chdir(_cwd)
print("CWD:", Path.cwd())


# Бенчмарк всего пайплайна на ground-truth парах

Задача: проверить, находит ли пайплайн `bi-encoder → cross-encoder` для описания товара тот пост в LanceDB, который размечен как релевантный.

Сравниваем две модели bi-encoder:
- **RoSBERTa** (384d, дообученный на серебряном датасете) — база `./lancedb_test_rosbert`
- **E5-base** (768d, из коробки) — база `./lancedb_test_e5`

Ground truth: 19 пар (описание товара → пост) из 10 написанных вручную постов. Эти 10 постов добавляются в обе базы, чтобы пайплайн искал их среди 50k+.

Метрики: **MRR**, **nDCG@K**, **Recall@K**, **P@K** для K ∈ {10, 50, 100}.


## 1. Импорты и настройки

In [15]:
import warnings
warnings.filterwarnings('ignore')

import json
import time
import math
from pathlib import Path

import numpy as np
import torch
import lancedb
import pyarrow as pa
from tqdm.auto import tqdm

import transformers
from transformers.modeling_utils import PreTrainedModel
transformers.PreTrainedModel = PreTrainedModel

from sentence_transformers import SentenceTransformer, CrossEncoder

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


Device: cpu


In [16]:
# ==================== НАСТРОЙКИ ====================

BI_ENCODER_ROSBERT_PATH = "models/final/bi-encoder"
BI_ENCODER_E5_PATH      = "intfloat/multilingual-e5-base"
CROSS_ENCODER_PATH      = "models/final/cross-encoder"

# Обе модели живут в одной папке lancedb_store, но в разных таблицах
LANCEDB_PATH    = "./lancedb_store"
TABLE_ROSBERT   = "rosberta-fine-tuned-50k"   # таблица для дообученного RoSBERTa (384d)
TABLE_E5        = "e5-base-base-50k"    # таблица для E5-base (768d)

GT_POSTS_JSON = "ground_truth_posts.json"
GT_PAIRS_JSON = "ground_truth_pairs.json"

# Маркеры для ground-truth постов
GT_CHANNEL = "gt"

# Параметры ретривала/ранжирования
TOP_K_RETRIEVE = 100    # сколько постов забирает bi-encoder
TOP_K_RERANK   = 100    # сколько из них пересортировывает cross-encoder
K_METRICS      = [10, 50, 100]


## 2. Загрузка ground truth

In [17]:
with open(GT_POSTS_JSON, encoding="utf-8") as f:
    gt_posts = json.load(f)

with open(GT_PAIRS_JSON, encoding="utf-8") as f:
    gt_pairs = json.load(f)

print(f"GT-постов: {len(gt_posts)}")
print(f"GT-пар (товар → пост): {len(gt_pairs)}")

# Индексы: post_num -> post_text, pair_index -> (query, relevant_post_nums)
post_text_by_num = {p["post_id"]: p["text"] for p in gt_posts}

# Один товар может быть размечен для нескольких постов — группируем по (imt_id, description)
from collections import defaultdict
query_to_rel_posts = defaultdict(set)
for pr in gt_pairs:
    query = pr["description"]
    query_to_rel_posts[query].add(pr["post_num"])

queries = list(query_to_rel_posts.keys())
print(f"Уникальных запросов: {len(queries)}")
for i, q in enumerate(queries, 1):
    rel = query_to_rel_posts[q]
    print(f"  {i}. [{len(rel)} пост(ов)] {q[:90]}...")


GT-постов: 10
GT-пар (товар → пост): 19
Уникальных запросов: 19
  1. [1 пост(ов)] Затирка для плитки готовая FABRIKK 0,8 кг, водостойкая смесь для швов с защитой от плесени...
  2. [1 пост(ов)] Имитация керамической плитки! Панели самоклеящиеся для стен влагостойкие глянцевые могут и...
  3. [1 пост(ов)] Развивающая книга "Пиши и стирай" от "Корпорация Шалостей" - это увлекательный подарок для...
  4. [1 пост(ов)] Представляем вам замечательную развивающую мозаику "Кораблик" от бренда "ДЕСЯТОЕ КОРОЛЕВСТ...
  5. [1 пост(ов)] Ищете эффективное средство для похудения? Fatburner Ultra Caps от PWR Ultimate Power — это...
  6. [1 пост(ов)] SLIMBERRY — натуральный и эффективный жиросжигатель, созданный для похудения без стресса и...
  7. [1 пост(ов)] Накидка на сиденье DongFeng Fengshen Yixuan GS (Донгфэнг Фенгшен Yixuan GS) — практичное р...
  8. [1 пост(ов)] Кроссовер Monjaro - премиальная модель Geely по уровню дизайна, материалов и технологий. Э...
  9. [1 пост(ов)] Представляем двуспальн

## 3. Загрузка моделей

In [18]:
print("Загружаем RoSBERTa (fine-tuned)...")
bi_rosbert = SentenceTransformer(BI_ENCODER_ROSBERT_PATH, device=device)
print(f"  dim={bi_rosbert.get_sentence_embedding_dimension()}")

print("\nЗагружаем E5-base (из коробки)...")
bi_e5 = SentenceTransformer(BI_ENCODER_E5_PATH, device=device)
print(f"  dim={bi_e5.get_sentence_embedding_dimension()}")

print("\nЗагружаем cross-encoder...")
cross_encoder = CrossEncoder(CROSS_ENCODER_PATH, device=device)
print("  готово")


Загружаем RoSBERTa (fine-tuned)...
  dim=384

Загружаем E5-base (из коробки)...
  dim=768

Загружаем cross-encoder...


The tokenizer you are loading from 'models/final/cross-encoder' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


  готово


## 4. Добавление 10 GT-постов в базу RoSBERTa

In [19]:
db = lancedb.connect(LANCEDB_PATH)
table_ros = db.open_table(TABLE_ROSBERT)
print(f"RoSBERTa-таблица ({TABLE_ROSBERT}): {table_ros.count_rows():,} записей до вставки")

# Идемпотентно: сначала удаляем все посты с channel='gt' (если были),
# потом добавляем ровно 10 свежих. Так дубликатов не будет при повторных прогонах.
before = table_ros.count_rows()
table_ros.delete(f"channel = '{GT_CHANNEL}'")
deleted = before - table_ros.count_rows()
print(f"  удалено старых GT-постов: {deleted}")

texts = [p["text"] for p in gt_posts]
embs = bi_rosbert.encode(texts, normalize_embeddings=True, show_progress_bar=False)

records = []
for p, emb in zip(gt_posts, embs):
    records.append({
        "vector": emb.tolist(),
        "text": p["text"],
        "channel": GT_CHANNEL,
        "category": "ground_truth",
    })
table_ros.add(records)
print(f"  добавлено GT-постов: {len(records)}")
print(f"Итого в RoSBERTa-таблице: {table_ros.count_rows():,}")

# Санити-чек: ровно 10 записей с channel='gt'
gt_count = table_ros.search(embs[0].tolist()).where(f"channel = '{GT_CHANNEL}'").limit(100).to_list()
assert len(gt_count) == len(gt_posts), f"ожидали {len(gt_posts)} GT-постов, нашли {len(gt_count)}"
print(f"Проверка: {len(gt_count)} GT-постов в таблице ✓")


RoSBERTa-таблица (posts): 50,010 записей до вставки
  удалено старых GT-постов: 10
  добавлено GT-постов: 10
Итого в RoSBERTa-таблице: 50,010
Проверка: 10 GT-постов в таблице ✓


## 5. Добавление 10 GT-постов в базу E5-base

In [20]:
table_e5 = db.open_table(TABLE_E5)
print(f"E5-таблица ({TABLE_E5}): {table_e5.count_rows():,} записей до вставки")

# Идемпотентно: удаляем старые GT-посты, добавляем 10 свежих
before = table_e5.count_rows()
table_e5.delete(f"channel = '{GT_CHANNEL}'")
deleted = before - table_e5.count_rows()
print(f"  удалено старых GT-постов: {deleted}")

# E5 требует префикс 'passage: ' при индексации документов
texts_e5 = ["passage: " + p["text"] for p in gt_posts]
embs_e5 = bi_e5.encode(texts_e5, normalize_embeddings=True, show_progress_bar=False)

records = []
for p, emb in zip(gt_posts, embs_e5):
    records.append({
        "vector": emb.tolist(),
        "text": p["text"],   # сам текст без префикса
        "channel": GT_CHANNEL,
        "category": "ground_truth",
    })
table_e5.add(records)
print(f"  добавлено GT-постов: {len(records)}")
print(f"Итого в E5-таблице: {table_e5.count_rows():,}")

# Санити-чек: ровно 10 записей с channel='gt'
gt_count = table_e5.search(embs_e5[0].tolist()).where(f"channel = '{GT_CHANNEL}'").limit(100).to_list()
assert len(gt_count) == len(gt_posts), f"ожидали {len(gt_posts)} GT-постов, нашли {len(gt_count)}"
print(f"Проверка: {len(gt_count)} GT-постов в таблице ✓")


E5-таблица (posts3): 50,000 записей до вставки
  удалено старых GT-постов: 0
  добавлено GT-постов: 10
Итого в E5-таблице: 50,010
Проверка: 10 GT-постов в таблице ✓


## 6. Функции метрик

In [21]:
def reciprocal_rank(ranks):
    """ranks — список рангов (1-индексация) первого релевантного результата для каждого запроса.
    Если для запроса не найдено ничего — рангом считается inf."""
    return np.mean([1.0 / r if r != float('inf') else 0.0 for r in ranks])


def recall_at_k(rel_positions, k):
    """rel_positions — список списков позиций релевантных постов для каждого запроса.
    Для каждого запроса: 1 если хотя бы одна позиция <= k, иначе 0."""
    hits = [1.0 if any(p <= k for p in positions) else 0.0 for positions in rel_positions]
    return np.mean(hits)


def precision_at_k(rel_positions, k):
    scores = [sum(1 for p in positions if p <= k) / k for positions in rel_positions]
    return np.mean(scores)


def ndcg_at_k(rel_positions, k):
    """Бинарная релевантность. Для каждого запроса:
       DCG = sum(1/log2(p+1)) по релевантным с позицией <= k.
       IDCG = sum(1/log2(i+1)) для i=1..min(num_relevant, k). """
    scores = []
    for positions in rel_positions:
        dcg = sum(1.0 / math.log2(p + 1) for p in positions if p <= k)
        num_rel = min(len(positions), k)
        idcg = sum(1.0 / math.log2(i + 1) for i in range(1, num_rel + 1))
        scores.append(dcg / idcg if idcg > 0 else 0.0)
    return np.mean(scores)


## 7. Функции пайплайна

Для каждой конфигурации прогоняем два варианта:
- **retrieve-only** — только bi-encoder, без cross-encoder;
- **retrieve+rerank** — bi-encoder достаёт TOP_K_RETRIEVE, cross-encoder пересортировывает топ TOP_K_RERANK.


In [22]:
def retrieve(table, query_vec, k):
    """Возвращает список словарей {text, channel}."""
    return (
        table.search(query_vec, query_type="vector")
        .limit(k)
        .select(["text", "channel"])
        .to_list()
    )


def rerank_with_cross_encoder(query, candidates, top_k):
    pairs = [[query, c["text"]] for c in candidates[:top_k]]
    scores = cross_encoder.predict(pairs, show_progress_bar=False)
    order = np.argsort(-scores)
    return [candidates[i] for i in order]


def positions_of_relevant(results, relevant_post_texts):
    """Возвращает список позиций (1-индексация) найденных релевантных постов в результатах."""
    positions = []
    for i, r in enumerate(results, start=1):
        if r["text"] in relevant_post_texts:
            positions.append(i)
    return positions


## 8. Прогон бенчмарка

In [23]:
def run_benchmark(model_name, encode_query_fn, table):
    """Гоняет обе конфигурации (retrieve-only / retrieve+rerank) на всех запросах.
    Возвращает словарь с метриками."""

    rel_positions_retrieve = []
    rel_positions_rerank = []
    first_rank_retrieve = []
    first_rank_rerank = []

    for query in tqdm(queries, desc=f"{model_name}", leave=False):
        relevant_post_nums = query_to_rel_posts[query]
        relevant_texts = {post_text_by_num[n] for n in relevant_post_nums}

        # --- retrieve ---
        qvec = encode_query_fn(query)
        retrieved = retrieve(table, qvec, TOP_K_RETRIEVE)

        pos_ret = positions_of_relevant(retrieved, relevant_texts)
        rel_positions_retrieve.append(pos_ret)
        first_rank_retrieve.append(min(pos_ret) if pos_ret else float('inf'))

        # --- rerank ---
        top_for_rerank = retrieved[:TOP_K_RERANK]
        reranked = rerank_with_cross_encoder(query, top_for_rerank, TOP_K_RERANK)
        # Оставшиеся (TOP_K_RETRIEVE..) остаются в исходном порядке после пересортированных
        final = reranked + retrieved[TOP_K_RERANK:]

        pos_rer = positions_of_relevant(final, relevant_texts)
        rel_positions_rerank.append(pos_rer)
        first_rank_rerank.append(min(pos_rer) if pos_rer else float('inf'))

    results = {}
    for label, rel_pos, ranks in [
        ("retrieve_only", rel_positions_retrieve, first_rank_retrieve),
        ("retrieve+rerank", rel_positions_rerank, first_rank_rerank),
    ]:
        m = {"MRR": reciprocal_rank(ranks)}
        for k in K_METRICS:
            m[f"Recall@{k}"] = recall_at_k(rel_pos, k)
            m[f"nDCG@{k}"]   = ndcg_at_k(rel_pos, k)
            m[f"P@{k}"]      = precision_at_k(rel_pos, k)
        results[label] = m

    return results, first_rank_retrieve, first_rank_rerank


In [24]:
# RoSBERTa: запрос кодируется как есть, без префикса
def encode_rosbert(q):
    return bi_rosbert.encode([q], normalize_embeddings=True)[0].tolist()

t0 = time.time()
ros_metrics, ros_ranks_ret, ros_ranks_rer = run_benchmark("RoSBERTa", encode_rosbert, table_ros)
print(f"RoSBERTa — {time.time()-t0:.1f}s")


RoSBERTa:   0%|          | 0/19 [00:00<?, ?it/s]

RoSBERTa — 674.6s


In [25]:
# E5: запрос с префиксом 'query: '
def encode_e5(q):
    return bi_e5.encode(["query: " + q], normalize_embeddings=True)[0].tolist()

t0 = time.time()
e5_metrics, e5_ranks_ret, e5_ranks_rer = run_benchmark("E5-base", encode_e5, table_e5)
print(f"E5-base — {time.time()-t0:.1f}s")


E5-base:   0%|          | 0/19 [00:00<?, ?it/s]

E5-base — 660.3s


## 9. Сводная таблица метрик

In [26]:
import pandas as pd

rows = []
for model_name, metrics in [("RoSBERTa (384d, ft)", ros_metrics), ("E5-base (768d, zero-shot)", e5_metrics)]:
    for stage, m in metrics.items():
        row = {"model": model_name, "stage": stage}
        row.update(m)
        rows.append(row)

df = pd.DataFrame(rows)
cols = ["model", "stage", "MRR"] + [c for k in K_METRICS for c in (f"Recall@{k}", f"nDCG@{k}", f"P@{k}")]
df = df[cols]
pd.set_option('display.float_format', '{:.3f}'.format)
df


,model,stage,MRR,Recall@10,nDCG@10,P@10,Recall@50,nDCG@50,P@50,Recall@100,nDCG@100,P@100
0,"RoSBERTa (384d, ft)",retrieve_only,0.018,0.053,0.026,0.005,0.053,0.026,0.001,0.105,0.035,0.001
1,"RoSBERTa (384d, ft)",retrieve+rerank,0.010,0.053,0.019,0.005,0.053,0.019,0.001,0.105,0.028,0.001
2,"E5-base (768d, zero-shot)",retrieve_only,0.064,0.105,0.068,0.011,0.211,0.092,0.004,0.263,0.100,0.003
3,"E5-base (768d, zero-shot)",retrieve+rerank,0.048,0.158,0.072,0.016,0.158,0.072,0.003,0.263,0.090,0.003


## 10. Ранги по запросам (для отладки)

In [27]:
details = []
for i, q in enumerate(queries):
    details.append({
        "query": q[:80] + ("..." if len(q) > 80 else ""),
        "ros_retrieve_rank": ros_ranks_ret[i] if ros_ranks_ret[i] != float('inf') else f">{TOP_K_RETRIEVE}",
        "ros_rerank_rank":   ros_ranks_rer[i] if ros_ranks_rer[i] != float('inf') else f">{TOP_K_RETRIEVE}",
        "e5_retrieve_rank":  e5_ranks_ret[i] if e5_ranks_ret[i] != float('inf') else f">{TOP_K_RETRIEVE}",
        "e5_rerank_rank":    e5_ranks_rer[i] if e5_ranks_rer[i] != float('inf') else f">{TOP_K_RETRIEVE}",
    })

details_df = pd.DataFrame(details)
details_df


,query,ros_retrieve_rank,ros_rerank_rank,e5_retrieve_rank,e5_rerank_rank
0,"Затирка для плитки готовая FABRIKK 0,8 кг, вод...",3,6,1,8
1,Имитация керамической плитки! Панели самоклеящ...,56,51,33,51
2,"Развивающая книга ""Пиши и стирай"" от ""Корпорац...",>100,>100,>100,>100
3,Представляем вам замечательную развивающую моз...,>100,>100,>100,>100
4,Ищете эффективное средство для похудения? Fatb...,>100,>100,>100,>100
5,SLIMBERRY — натуральный и эффективный жиросжиг...,>100,>100,>100,>100
6,Накидка на сиденье DongFeng Fengshen Yixuan GS...,>100,>100,87,2
7,Кроссовер Monjaro - премиальная модель Geely п...,>100,>100,13,4
8,Представляем двуспальный надувной матрас 203Х1...,>100,>100,>100,>100
9,БЕЗ побочных эффектов! Препарат безопасен и не...,>100,>100,>100,>100


## Что дальше

- Если RoSBERTa сильно лучше на retrieve — дообучение сработало, это ожидаемо.
- Если E5-base не сильно хуже — имеет смысл дообучить E5 на серебряном датасете, он глубже (768d, 12 слоёв).
- Если cross-encoder не улучшает retrieve — проверить, что `models/final/cross-encoder` грузится правильно (это дообученная `DiTy/cross-encoder-russian-msmarco`).

Для расширения бенчмарка:
- Разметить больше пар (текущие 19 пар — маленькая выборка).
- Добавить BM25 как третью точку сравнения (использовать FTS-индекс из `create-lancedb-v2`).
